In [ ]:
# Import core libraries for data handling, modeling, tuning, and evaluation.
import pandas as pd
import joblib
import optuna
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE
import seaborn as sns
import matplotlib.pyplot as plt


NameError: name 'best_model' is not defined

In [ ]:
# Silence non-critical warnings for cleaner notebook output.
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:
# Load the cardiovascular dataset and preview shape/sample rows.
df = pd.read_csv("../data/cardiovascular.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
# Plot class distribution to check target balance visually.
plt.figure()
sns.countplot(x=df['risk_category'])
plt.title("Cardiovascular Class Distribution")
plt.show()


In [ ]:
# Inspect missing values and raw target labels.
print("\nMissing Values:\n", df.isnull().sum())
print("\nTarget Values:\n", df["risk_category"].unique())


In [ ]:
# Clean cardiovascular target labels and map them to strict binary classes.
df.drop(columns=["Patient_ID", "heart_disease_risk_score"], inplace=True)
df.dropna(inplace=True)
df["risk_category"] = df["risk_category"].astype(str).str.strip().str.lower()

# Strict binary mapping (dataset supports only low / high)
valid_labels = {"low", "high"}
found_labels = set(df["risk_category"].unique())
unexpected = sorted(found_labels - valid_labels)
if unexpected:
    raise ValueError(f"Unexpected target labels found: {unexpected}. Expected only {sorted(valid_labels)}")

mapping = {
    "low": 0,
    "high": 1,
}

df["risk_category"] = df["risk_category"].map(mapping).astype(int)

print("After Mapping (0=Low Risk, 1=High Risk):", df["risk_category"].value_counts())


In [ ]:
# Split cardiovascular features/target into train, validation, and test sets.
X = df.drop("risk_category", axis=1)
y = df["risk_category"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


In [ ]:
# preprocessing pipeline (scaling + encoding) and transform feature matrices.
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns: {numeric_cols}")
print(f"Categorical columns: {categorical_cols}")

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

X_train_prep = preprocessor.fit_transform(X_train)
X_val_prep = preprocessor.transform(X_val)
X_test_prep = preprocessor.transform(X_test)

feature_names_after = numeric_cols + list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols))
print(f"Features after preprocessing: {len(feature_names_after)}")
print(f"Split sizes -> train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}")


In [ ]:
# SelectKBest feature selection
print("\nFinding Best K Value...")

k_values = [5, 8, 10, 12, 15]

best_k = None
best_score = 0

for k in k_values:

    temp_selector = SelectKBest(f_classif, k=k)

    X_train_temp = temp_selector.fit_transform(X_train_prep, y_train)
    X_val_temp = temp_selector.transform(X_val_prep)

    # Apply SMOTE strictly for evaluating balanced baseline performance
    smote_temp = SMOTE(random_state=42)
    X_train_temp_sm, y_train_sm = smote_temp.fit_resample(X_train_temp, y_train)

    temp_model = RandomForestClassifier(random_state=42)
    temp_model.fit(X_train_temp_sm, y_train_sm)

    preds = temp_model.predict(X_val_temp)

    score = f1_score(y_val, preds)

    print(f"K = {k} --> F1 Score = {score:.4f}")

    if score > best_score:
        best_score = score
        best_k = k

print(f"\nBest K Selected: {best_k}")

selector = SelectKBest(f_classif, k=best_k)

X_train_prep = selector.fit_transform(X_train_prep, y_train)
X_val_prep = selector.transform(X_val_prep)
X_test_prep = selector.transform(X_test_prep)

selected_idx = selector.get_support(indices=True)

selected_features = [
    feature_names_after[i]
    for i in selected_idx
]

print("\nFeature Selection Complete:")
print(f"Selected {len(selected_features)} features:")
print(selected_features)


In [ ]:
# Apply SMOTE on training data to reduce class imbalance.
smote = SMOTE(random_state=42)
X_train_prep, y_train = smote.fit_resample(X_train_prep, y_train)
print("SMOTE applied - training data balanced")


In [ ]:
# Train baseline models and compare validation performance.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42, n_jobs=-1),
    "SVM": SVC(probability=True, random_state=42)
}

baseline_results = []
for name, model in models.items():
    model.fit(X_train_prep, y_train)
    y_val_pred = model.predict(X_val_prep)

    baseline_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_val_pred),
        "Precision": precision_score(y_val, y_val_pred),
        "Recall": recall_score(y_val, y_val_pred),
        "F1": f1_score(y_val, y_val_pred),
    })

baseline_results_df = pd.DataFrame(baseline_results)[["Model", "Accuracy", "Precision", "Recall", "F1"]]
print("\n-- Baseline Model Comparison (Validation) --")
display(baseline_results_df.sort_values("F1", ascending=False).reset_index(drop=True))


In [ ]:
# Configure cross-validation and define a helper for consistent CV reporting.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def print_cv_summary(model_name, cv_scores):
    fold_scores = [round(float(s), 4) for s in cv_scores]
    mean_f1 = float(cv_scores.mean())
    std_f1 = float(cv_scores.std())
    print(f"{model_name} CV fold F1:", fold_scores)
    return mean_f1, std_f1


In [ ]:
# Run Optuna hyperparameter tuning in a simple single-loop workflow.
optuna.logging.set_verbosity(optuna.logging.WARNING)

model_tuning_plan = {
    "Logistic Regression": {"n_trials": 20},
    "Random Forest": {"n_trials": 20},
    "XGBoost": {"n_trials": 25},
    "SVM": {"n_trials": 25},
}

optimized_models = {}
cv_scores_all = {}
best_params_all = {}


def make_objective(model_name, X_train, y_train, cv):
    def objective(trial):
        if model_name == "Logistic Regression":
            params = {
                "C": trial.suggest_float("C", 1e-4, 1e2, log=True),
                "solver": trial.suggest_categorical("solver", ["liblinear", "lbfgs", "saga"]),
                "max_iter": 1000,
                "random_state": 42,
            }
            model = LogisticRegression(**params)
        elif model_name == "Random Forest":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 600),
                "max_depth": trial.suggest_int("max_depth", 3, 30),
                "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                "random_state": 42,
                "n_jobs": -1,
            }
            model = RandomForestClassifier(**params)
        elif model_name == "XGBoost":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 800),
                "max_depth": trial.suggest_int("max_depth", 2, 12),
                "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "random_state": 42,
                "eval_metric": "logloss",
                "n_jobs": -1,
            }
            model = XGBClassifier(**params)
        else:  # SVM
            kernel = trial.suggest_categorical("kernel", ["linear", "rbf"])
            params = {
                "C": trial.suggest_float("C", 1e-4, 1e2, log=True),
                "kernel": kernel,
                "probability": True,
                "random_state": 42,
            }
            if kernel == "rbf":
                params["gamma"] = trial.suggest_float("gamma", 1e-4, 1e0, log=True)
            model = SVC(**params)

        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
        return float(scores.mean())
    return objective

for model_name, cfg in model_tuning_plan.items():
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(
        make_objective(model_name, X_train_prep, y_train, cv),
        n_trials=cfg["n_trials"],
    )
    best_params_all[model_name] = study.best_params

    if model_name == "Logistic Regression":
        tuned_model = LogisticRegression(**study.best_params, max_iter=1000, random_state=42)
    elif model_name == "Random Forest":
        tuned_model = RandomForestClassifier(**study.best_params, random_state=42, n_jobs=-1)
    elif model_name == "XGBoost":
        tuned_model = XGBClassifier(**study.best_params, random_state=42, eval_metric="logloss", n_jobs=-1)
    else:
        tuned_model = SVC(**study.best_params, probability=True, random_state=42)

    tuned_model.fit(X_train_prep, y_train)
    optimized_models[model_name] = tuned_model

    scores = cross_val_score(tuned_model, X_train_prep, y_train, cv=cv, scoring="f1", n_jobs=-1)
    cv_scores_all[model_name] = scores
    print_cv_summary(model_name, scores)

optimized_lr = optimized_models["Logistic Regression"]
optimized_rf = optimized_models["Random Forest"]
optimized_xgb = optimized_models["XGBoost"]
optimized_svm = optimized_models["SVM"]

lr_cv_scores = cv_scores_all["Logistic Regression"]
rf_cv_scores = cv_scores_all["Random Forest"]
xgb_cv_scores = cv_scores_all["XGBoost"]
svm_cv_scores = cv_scores_all["SVM"]


In [ ]:
# Print consolidated 5-fold CV mean +/- std for all tuned models.
cv_scores_all = {
    "Logistic Regression": lr_cv_scores,
    "Random Forest": rf_cv_scores,
    "XGBoost": xgb_cv_scores,
    "SVM": svm_cv_scores,
}

print("\n5-Fold CV Mean +/- Std (All Models):")
name_width = max(len(name) for name in cv_scores_all)
for model_name, scores in cv_scores_all.items():
    mean_f1 = float(scores.mean())
    std_f1 = float(scores.std())
    print(f"  {model_name:<{name_width}} : {mean_f1:.4f} +/- {std_f1:.4f}")


In [ ]:
# Compare optimized models on validation metrics.
optimized_models = {
    "Logistic Regression": optimized_lr,
    "Random Forest": optimized_rf,
    "XGBoost": optimized_xgb,
    "SVM": optimized_svm,
}

optimized_results = []
for name, model in optimized_models.items():
    y_val_pred = model.predict(X_val_prep)

    optimized_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_val_pred),
        "Precision": precision_score(y_val, y_val_pred),
        "Recall": recall_score(y_val, y_val_pred),
        "F1": f1_score(y_val, y_val_pred),
    })

optimized_results_df = pd.DataFrame(optimized_results)[["Model", "Accuracy", "Precision", "Recall", "F1"]]
print("\n-- Optimized Model Comparison (Validation) --")
display(optimized_results_df.sort_values("F1", ascending=False).reset_index(drop=True))

# Plot bar chart for Optimized Model Comparison (Validation)
optimized_results_df.set_index("Model").plot(kind="bar", figsize=(10, 5))
plt.title("Optimized Model Comparison (Validation)")
plt.ylabel("Score")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# -- Threshold Tuning --
threshold_candidates = [round(i * 0.05, 2) for i in range(6, 15)]

val_probs_lr = optimized_lr.predict_proba(X_val_prep)[:, 1]
best_threshold_lr = 0.5
best_f1_lr = f1_score(y_val, (val_probs_lr >= 0.5).astype(int))
for t in threshold_candidates:
    score = f1_score(y_val, (val_probs_lr >= t).astype(int))
    if score > best_f1_lr:
        best_f1_lr, best_threshold_lr = score, t
print(f"Logistic Regression : best threshold = {best_threshold_lr}, F1 = {best_f1_lr:.4f}")

val_probs_rf = optimized_rf.predict_proba(X_val_prep)[:, 1]
best_threshold_rf = 0.5
best_f1_rf = f1_score(y_val, (val_probs_rf >= 0.5).astype(int))
for t in threshold_candidates:
    score = f1_score(y_val, (val_probs_rf >= t).astype(int))
    if score > best_f1_rf:
        best_f1_rf, best_threshold_rf = score, t
print(f"Random Forest       : best threshold = {best_threshold_rf}, F1 = {best_f1_rf:.4f}")

val_probs_xgb = optimized_xgb.predict_proba(X_val_prep)[:, 1]
best_threshold_xgb = 0.5
best_f1_xgb = f1_score(y_val, (val_probs_xgb >= 0.5).astype(int))
for t in threshold_candidates:
    score = f1_score(y_val, (val_probs_xgb >= t).astype(int))
    if score > best_f1_xgb:
        best_f1_xgb, best_threshold_xgb = score, t
print(f"XGBoost             : best threshold = {best_threshold_xgb}, F1 = {best_f1_xgb:.4f}")

val_probs_svm = optimized_svm.predict_proba(X_val_prep)[:, 1]
best_threshold_svm = 0.5
best_f1_svm = f1_score(y_val, (val_probs_svm >= 0.5).astype(int))
for t in threshold_candidates:
    score = f1_score(y_val, (val_probs_svm >= t).astype(int))
    if score > best_f1_svm:
        best_f1_svm, best_threshold_svm = score, t
print(f"SVM                 : best threshold = {best_threshold_svm}, F1 = {best_f1_svm:.4f}")


In [ ]:
# Select the best optimized model using threshold-adjusted validation F1.
tuning_results = {
    "Logistic Regression": (optimized_lr, best_threshold_lr, best_f1_lr),
    "Random Forest": (optimized_rf, best_threshold_rf, best_f1_rf),
    "XGBoost": (optimized_xgb, best_threshold_xgb, best_f1_xgb),
    "SVM": (optimized_svm, best_threshold_svm, best_f1_svm),
}

best_model_name = max(tuning_results, key=lambda k: tuning_results[k][2])
best_model, threshold, best_threshold_f1 = tuning_results[best_model_name]

print(f"Best Model : {best_model_name}")
print(f"Threshold  : {threshold}")
print(f"Val F1     : {best_threshold_f1:.4f}")


In [ ]:
# Evaluate final model on test set and plot confusion matrix.
y_test_prob = best_model.predict_proba(X_test_prep)[:, 1]
y_test_pred = (y_test_prob >= threshold).astype(int)
cm_test = confusion_matrix(y_test, y_test_pred)

print(f"\n-- Final Test Performance [{best_model_name} | threshold={threshold}] --")
print(f"Accuracy  : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_test_pred):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_test_pred):.4f}")

plt.figure(figsize=(6, 4))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


In [ ]:
# Artifact Saving
final_artifact = {
    "model": best_model,
    "preprocessor": preprocessor,
    "selector": selector,
    "threshold": threshold,
    "selected_features": selected_features,
    "feature_columns": list(X.columns),
    "features": list(X.columns),
    "model_name": best_model_name,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "positive_class": 1,
    "negative_class": 0,
    "class_label_map": {0: "Low Risk", 1: "High Risk"},
}

joblib.dump(final_artifact, "../models/cardiovascular_model.pkl")

print("\nCardiovascular model saved successfully!")
print(f"Model: {best_model_name}")
print(f"Threshold: {threshold}")
print(f"Selector: SelectKBest(k={best_k})")
print("Preprocessor: StandardScaler + OneHotEncoder")
